# OSRM for vehicle routing (VRP / TSP)

This notebook is a **focused guide** to [OSRM](https://project-osrm.org/) (Open Source Routing Machine): what it does, which HTTP APIs matter for **your** project (`routes.py`, clustering, distance matrices), and how to call them from Python.

**Assumption:** you already have `osrm-routed` running (e.g. `http://127.0.0.1:5000`) with a dataset that covers your delivery coordinates (e.g. UK extract).

## What you will learn

1. **Coordinate order** OSRM uses: `longitude,latitude` (not lat,lon).
2. **`/route/v1`** — one path between ordered waypoints (depot → stop → …).
3. **`/table/v1`** — many-to-many **durations** or **distances** (the heart of replacing Haversine in NN / 2-opt).
4. **How this maps to your repo**: build a cost matrix → same TSP heuristics, better realism.
5. **Reliability gotchas**: `NoRoute`, rate limits, extract coverage, `annotations` vs `overview`.

## 1. What OSRM actually does

- OSRM builds a **road graph** from OpenStreetMap and runs **shortest-path** queries on it.
- It exposes a **JSON HTTP API** (not “a Python library”): your code uses `urllib`, `requests`, or `httpx`.
- **Profiles** (e.g. `driving`, `car`) encode rules: speeds, one-ways, turn restrictions — depending on how the server was built (`car.lua`, etc.).

For **route optimisation** you rarely need turn-by-turn geometry at first; you need **consistent numeric costs** between pairs of stops (time or distance).

In [2]:
import json
import urllib.parse
import urllib.request

# Change if your server listens elsewhere
OSRM_BASE = "http://127.0.0.1:5000"
PROFILE = "driving"  # must match how osrm-routed was started

## 2. Coordinate order: `lon,lat` everywhere in the URL

OSRM path coordinates look like:

`{lon1},{lat1};{lon2},{lat2};...`

Your CSV has `latitude`, `longitude` — **swap when building URLs**.

In [3]:
def lonlat_pair(lat: float, lon: float) -> str:
    """OSRM expects longitude first."""
    return f"{lon},{lat}"


def osrm_get(path: str, params=None) -> dict:
    q = urllib.parse.urlencode(params or {})
    url = f"{OSRM_BASE}{path}"
    if q:
        url = f"{url}?{q}"
    with urllib.request.urlopen(url, timeout=60) as resp:
        return json.loads(resp.read().decode("utf-8"))

## 3. `/route/v1` — single path through waypoints

**Use when:** you want one continuous route (polyline, duration, distance) for an **ordered** list: depot → A → B → depot.

**Useful for:** debugging, map visualisation, comparing “OSRM leg” vs Haversine.

**Not enough alone for NN / 2-opt:** those algorithms need **every pair** (i,j) of costs — that is `/table`.

In [4]:
# Example: depot → first delivery (from data/deliveries.csv schema)
# DEPOT: 52.527037, -1.390319  |  D001: 52.642145, -1.123577  (replace with your CSV row)
depot_lat, depot_lon = 52.527037, -1.390319
stop_lat, stop_lon = 52.642145, -1.123577

coords = ";".join(
    [lonlat_pair(depot_lat, depot_lon), lonlat_pair(stop_lat, stop_lon)]
)
path = f"/route/v1/{PROFILE}/{coords}"
data = osrm_get(path, {"overview": "false", "steps": "false"})

assert data.get("code") == "Ok", data
leg = data["routes"][0]
print("distance (m):", leg["distance"])
print("duration (s):", leg["duration"])

distance (m): 25412.1
duration (s): 1503.5


## 4. `/table/v1` — matrix API (what you need for TSP / VRP)

**Returns:** for many sources × many destinations:

- **`durations`** — seconds (default matrix metric)
- **`distances`** — metres (when requested via `annotations`)

**Typical pattern for your project:**

1. Collect all stops in one cluster (plus depot): indices `0..n-1`.
2. One `table` call with `sources=all&destinations=all` (or batched if n is huge).
3. Build `cost[i][j]` = duration or distance.
4. In code, replace `haversine(a,b)` with `cost[id(a)][id(b)]` (and handle `null` = no road connection).

**Docs:** [OSRM HTTP API — Table service](https://github.com/Project-OSRM/osrm-backend/blob/master/docs/http.md#table-service)

In [5]:
# Small 3-point example: depot + two stops (same coords as above + second fake point)
p0 = lonlat_pair(52.527037, -1.390319)
p1 = lonlat_pair(52.642145, -1.123577)
p2 = lonlat_pair(52.620365, -1.137335)
coord_path = f"/table/v1/{PROFILE}/{p0};{p1};{p2}"

table = osrm_get(
    coord_path,
    {
        "annotations": "duration,distance",
        "sources": "0;1;2",
        "destinations": "0;1;2",
    },
)

assert table.get("code") == "Ok", table
durations = table["durations"]  # seconds, shape n×n
distances = table["distances"]  # metres, may contain null if unreachable
print("durations[s] row 0:", durations[0])
print("distances[m] row 0:", distances[0])

durations[s] row 0: [0, 1503.5, 1354.3]
distances[m] row 0: [0, 25412.1, 23852.5]


## 5. Link to *your* repository

| Piece | Role |
|-------|------|
| `data/deliveries.csv` | Inputs: `latitude`, `longitude` per stop |
| `cluster2.py` | Groups stops into vans (today: sklearn on lat/lon; optional future: road-distance clustering) |
| `routes.py` | Orders stops per van with NN / 2-opt / RR-2-opt — **swap distance** to OSRM matrix for road realism |

**Minimal integration path (recommended):**

1. Keep clustering as-is OR upgrade later.
2. For each cluster, build **one** OSRM table for `{depot} ∪ cluster_stops`.
3. Cache JSON to `data/osrm_cache_cluster_<id>.json` to avoid re-querying while tuning algorithms.

**Cost choice:**

- **Duration** — better if you care about “time on road” / traffic-like weighting (still static without live traffic).
- **Distance** — simpler to explain in a report (km).

## 6. Reliability: what breaks in real use

| Issue | What to do |
|--------|------------|
| `NoRoute` / `null` in matrix | Snapped coordinate off network; try `/nearest` first, or check extract covers the point |
| Huge `n` → URL too long | Batch `table` requests (chunks of coordinates) |
| Slow first query | Server cold-start; UK graph is large — allow RAM, SSD |
| Different results after rebuild | Map data / OSRM version changed; pin versions for reproducibility |

**Nearest service** (snap a point to the road graph):  
`GET /nearest/v1/{profile}/{lon},{lat}` — useful before building matrices.

In [ ]:
# Optional: load depot + first data row from repo (requires running from VRP repo or adjust REPO_ROOT)
from pathlib import Path
import csv

REPO_ROOT = Path("..").resolve()  # osrm-learning/ sits next to data/ under VRP
csv_path = REPO_ROOT / "data" / "deliveries.csv"
if csv_path.exists():
    with csv_path.open(newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    depot = next(r for r in rows if r["delivery_id"] == "DEPOT")
    stop = next(r for r in rows if r["delivery_id"] != "DEPOT")
    d_lat, d_lon = float(depot["latitude"]), float(depot["longitude"])
    s_lat, s_lon = float(stop["latitude"]), float(stop["longitude"])
    coords = ";".join([lonlat_pair(d_lat, d_lon), lonlat_pair(s_lat, s_lon)])
    r = osrm_get(f"/route/v1/{PROFILE}/{coords}", {"overview": "simplified"})
    print("CSV route:", depot["delivery_id"], "->", stop["delivery_id"], "| code:", r.get("code"))
    if r.get("code") == "Ok":
        print("distance_m:", r["routes"][0]["distance"], "duration_s:", r["routes"][0]["duration"])
else:
    print("Skip: not found", csv_path)

CSV route: DEPOT -> D001 | code: Ok
distance_m: 25412.1 duration_s: 1503.5


: 

## 7. Further reading (official)

- [OSRM HTTP API documentation](https://github.com/Project-OSRM/osrm-backend/blob/master/docs/http.md)
- [Running OSRM](https://github.com/Project-OSRM/osrm-backend/wiki/Running-OSRM) (extract → partition → customize → routed)

---

*Notebook lives in `VRP/osrm-learning/` — keep OSRM server separate; this file documents the contract your Python code should use.*